# multi_strat_053

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (90).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 1 |
| Detected functions | process_ticker, backtest_trades, calculate_quarterly_hpr, safe_profit_factor |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from tqdm import tqdm
from joblib import Parallel, delayed
import plotly.graph_objects as go
import argparse

# Install required packages (uncomment if running in a new environment)
# !pip install cvxpy xgboost

# Parse command-line arguments for file paths
parser = argparse.ArgumentParser(description="Walk-forward backtesting script")
parser.add_argument('--ohlcv_path', default='5min_data.csv', help='Path to OHLCV data')
parser.add_argument('--tradebook_path', default='tradebook.csv', help='Path to tradebook data')
parser.add_argument('--data_path', default='5min_data_1.csv', help='Path to additional data')
args = parser.parse_args()

# Load datasets
ohlcv = pd.read_csv(args.ohlcv_path)  # Feb 2015 - Nov 2024
tradebook = pd.read_csv(args.tradebook_path)  # 8-year tradebook
data = pd.read_csv(args.data_path)  # Update to cover 2015-2024

# Convert date columns to datetime
ohlcv['Date'] = pd.to_datetime(ohlcv['Date'])
tradebook['Entry Time'] = pd.to_datetime(tradebook['Entry Time'])
tradebook['Exit Time'] = pd.to_datetime(tradebook['Exit Time'])
data['Date'] = pd.to_datetime(data['Date'])

# Ensure 'signal' column exists in data
if 'signal' not in data.columns:
    print("Warning: 'signal' column missing in data. Assuming no signals.")
    data['signal'] = 0  # Default to no trades

def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time, prev_cumulative_pnl=0):
    open_positions = {}
    trades = []
    cumulative_pnl = prev_cumulative_pnl

    for row in ticker_data.itertuples():
        timestamp, close, open_price, high, low, signal = row.Date, row.close, row.open, row.high, row.low, getattr(row, 'signal', 0)

        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif timestamp.strftime("%H:%M") >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl
                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        if ticker not in open_positions and timestamp.strftime("%H:%M") < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

def backtest_trades(data, prev_cumulative_pnl=0):
    initial_capital = 100000
    brokerage = 0.001
    slippage = 0.0001
    tp_pct = 0.02
    sl_pct = 0.005
    end_time = "15:25"
    stop_trading_time = "15:15"

    unique_tickers = data['Ticker'].unique()
    results = Parallel(n_jobs=-1)(delayed(process_ticker)(ticker, data[data['Ticker'] == ticker],
                                                         initial_capital, brokerage, slippage, tp_pct, sl_pct,
                                                         end_time, stop_trading_time, prev_cumulative_pnl)
                                 for ticker in tqdm(unique_tickers))

    trades = [trade for result in results for trade in result]
    if not trades:
        return pd.DataFrame(columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
                                     "PnL", "Exit Reason", "Cumulative PnL", "Direction"])
    return pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
                                        "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# Function to calculate quarterly HPR
def calculate_quarterly_hpr(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Quarter"] = df["Date"].dt.quarter
    quarterly_returns = (
        df.groupby(["Ticker", "Year", "Quarter"])["close"]
        .agg(lambda x: (x.iloc[-1] - x.iloc[0]) / x.iloc[0] if len(x) > 0 else 0)
        .reset_index()
        .rename(columns={"close": "HPR"})
    )
    return quarterly_returns

# Safe Profit Factor Calculation
def safe_profit_factor(x):
    pos_sum = x[x > 0].sum()
    neg_sum = abs(x[x < 0].sum())
    if neg_sum < 1e-10 and pos_sum < 1e-10:
        return 0  # No profits or losses
    return pos_sum / neg_sum if neg_sum > 1e-10 else float('inf') if pos_sum > 0 else 0

# Initial training period: Jan 2015 - Dec 2016
initial_train_start = pd.to_datetime("2015-01-01")
initial_train_end = pd.to_datetime("2016-12-31")
initial_ohlcv = ohlcv[(ohlcv["Date"] >= initial_train_start) & (ohlcv["Date"] <= initial_train_end)]
initial_tradebook = tradebook[(tradebook["Entry Time"] >= initial_train_start) &
                              (tradebook["Entry Time"] <= initial_train_end)]

# Quarterly periods from Q1 2017 to Q4 2024
quarters = pd.date_range(start="2017-01-01", end="2024-12-31", freq="QS")
all_tradebooks = []
prev_cumulative_pnl = 0

for i in range(len(quarters)):
    train_end = quarters[i] - pd.offsets.Day(1)  # End of previous quarter
    test_start = quarters[i]
    test_end = test_start + pd.offsets.QuarterEnd()

    # Training window: Expands each quarter
    train_start = initial_train_start
    train_ohlcv = ohlcv[(ohlcv["Date"] >= train_start) & (ohlcv["Date"] <= train_end)]
    test_ohlcv = ohlcv[(ohlcv["Date"] >= test_start) & (ohlcv["Date"] <= test_end)]
    train_tradebook = tradebook[(tradebook["Entry Time"] >= train_start) &
                                (tradebook["Entry Time"] <= train_end)].copy()

    # Calculate HPR and Strategy Returns
    train_hpr = calculate_quarterly_hpr(train_ohlcv)
    train_tradebook["Entry Year"] = train_tradebook["Entry Time"].dt.year
    train_tradebook["Entry Quarter"] = train_tradebook["Entry Time"].dt.quarter
    strategy_returns = (
        train_tradebook.groupby(["Ticker", "Entry Year", "Entry Quarter"])["PnL"]
        .sum()
        .reset_index()
        .rename(columns={"PnL": "Strategy_Return"})
    )

    # Merge data
    merged_data = train_hpr.merge(strategy_returns,
                                 left_on=["Ticker", "Year", "Quarter"],
                                 right_on=["Ticker", "Entry Year", "Entry Quarter"],
                                 how="left")
    merged_data.drop(columns=["Entry Year", "Entry Quarter"], inplace=True)
    merged_data.fillna(0, inplace=True)

    # Feature Engineering with robust handling
    summary = train_tradebook.groupby("Ticker").agg(
        total_pnl=("PnL", "sum"),
        total_trades=("PnL", "count"),
        win_rate=("PnL", lambda x: (x > 0).sum() / len(x) if len(x) > 0 else 0),
        max_drawdown=("Cumulative PnL", lambda x: (x - x.cummax()).min() if len(x) > 0 else 0),
        profit_factor=("PnL", safe_profit_factor)
    ).reset_index()
    summary.replace([np.inf, -np.inf], 0, inplace=True)
    summary["profit_factor"] = summary["profit_factor"].clip(upper=1e6, lower=-1e6)
    summary["max_drawdown"] = summary["max_drawdown"].clip(upper=1e6, lower=-1e6)

    merged_data = merged_data.merge(summary, on="Ticker", how="left").fillna(0)

    # Align features for training and testing
    features = ["HPR", "total_pnl", "total_trades", "win_rate", "max_drawdown", "profit_factor"]
    merged_data.dropna(subset=features + ["Strategy_Return"], inplace=True)
    X_train = merged_data[features]
    y_train = merged_data["Strategy_Return"]
    test_hpr = calculate_quarterly_hpr(test_ohlcv)
    X_test = test_hpr.merge(summary, on="Ticker", how="left").fillna(0)[features]

    # Handle infinities and extreme values
    X_train.replace([np.inf, -np.inf], 0, inplace=True)
    X_train = X_train.clip(upper=1e6, lower=-1e6)
    X_test.replace([np.inf, -np.inf], 0, inplace=True)
    X_test = X_test.clip(upper=1e6, lower=-1e6)
    X_test.dropna(inplace=True)

    # Check if training data is valid
    if X_train.empty or y_train.empty:
        print(f"Skipping training for {test_start.strftime('%Y-%m')} - No valid training data.")
        all_tradebooks.append(pd.DataFrame())
        continue

    # Train and predict
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    predicted_returns = model.predict(X_test)

    test_hpr["Predicted_Strategy_Return"] = predicted_returns
    top_10_stocks = test_hpr.nlargest(10, "Predicted_Strategy_Return")["Ticker"].tolist()
    print(f"Top 10 Stocks for {test_start.strftime('%Y-%m')} to {test_end.strftime('%Y-%m')}: {top_10_stocks}")

    # Backtest on top 10 stocks
    test_data = data[(data["Ticker"].isin(top_10_stocks)) & (data["Date"] >= test_start) & (data["Date"] <= test_end)].copy()
    if test_data.empty:
        print(f"Warning: No data available for backtesting in {test_start.strftime('%Y-%m')} to {test_end.strftime('%Y-%m')}. Skipping this quarter.")
        all_tradebooks.append(pd.DataFrame())
        continue

    quarter_tradebook = backtest_trades(test_data, prev_cumulative_pnl)
    if not quarter_tradebook.empty:
        prev_cumulative_pnl = quarter_tradebook["Cumulative PnL"].iloc[-1] if "Cumulative PnL" in quarter_tradebook.columns else prev_cumulative_pnl
    all_tradebooks.append(quarter_tradebook)

# Combine all quarterly tradebooks, filtering out empty DataFrames
final_tradebook = pd.concat([tb for tb in all_tradebooks if not tb.empty], ignore_index=True)
final_tradebook.to_csv("tradebook_walk_forward_8years.csv", index=False)
print("Final Tradebook Shape:", final_tradebook.shape)

# Visualize last quarter (Q4 2024)
if all_tradebooks and not all_tradebooks[-1].empty:
    last_quarter_tradebook = all_tradebooks[-1]
    ticker = last_quarter_tradebook["Ticker"].iloc[0]
    data_filtered = data[(data["Ticker"] == ticker) & (data["Date"] >= pd.to_datetime("2024-10-01")) &
                         (data["Date"] <= pd.to_datetime("2024-12-31"))]
    tradebook_filtered = last_quarter_tradebook[last_quarter_tradebook["Ticker"] == ticker]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=data_filtered["Date"], y=data_filtered["close"], mode='lines', name='Close Price', line=dict(color='blue')))

    added_legend = {"Long Entry": False, "Short Entry": False, "Exit": False, "Profit Trade": False, "Loss Trade": False}

    for _, trade in tradebook_filtered.iterrows():
        entry_time, exit_time = trade["Entry Time"], trade["Exit Time"]
        entry_price, exit_price = trade["Entry Price"], trade["Exit Price"]
        pnl, direction = trade["PnL"], trade["Direction"]

        entry_color, symbol = ("green", "triangle-up") if direction == "long" else ("red", "triangle-down")
        name_entry = "Long Entry" if direction == "long" else "Short Entry"
        trade_color = "green" if pnl > 0 else "red"
        trade_name = "Profit Trade" if pnl > 0 else "Loss Trade"

        fig.add_trace(go.Scatter(x=[entry_time], y=[entry_price], mode='markers',
                                marker=dict(color=entry_color, size=10, symbol=symbol),
                                name=name_entry, showlegend=not added_legend[name_entry]))
        added_legend[name_entry] = True

        fig.add_trace(go.Scatter(x=[exit_time], y=[exit_price], mode='markers',
                                marker=dict(color='black', size=10, symbol='x'),
                                name="Exit", showlegend=not added_legend["Exit"]))
        added_legend["Exit"] = True

        fig.add_trace(go.Scatter(x=[entry_time, exit_time], y=[entry_price, exit_price], mode='lines',
                                line=dict(color=trade_color, width=2, dash='dash'),
                                name=trade_name, showlegend=not added_legend[trade_name]))
        added_legend[trade_name] = True

    fig.update_layout(title=f"Trade Visualization for {ticker} (Q4 2024)", xaxis_title="Time", yaxis_title="Price",
                     legend_title="Legend", template="plotly_dark")
    fig.show()
else:
    print("No trades available to visualize for Q4 2024.")